# SAE Steering from Robust Features (Colab GPU)

Этот ноутбук — **Colab-аналог** `probes_experiment/run_sae_steering_from_features.py`.

**Что нужно загрузить:**
1. **test.csv** — CSV с колонками `question`, `verbal_uncertainty`, `sentence_semantic_entropy`.
2. **stats JSON** — файл `sae_feature_analysis_merged_stats.json` (результат `sae_feature_analysis_robust.py`).

Оба файла можно загрузить через виджет (секция 3) или указать путь на Google Drive (секция 2).

**Результат:** JSONL с полями `alpha`, `question`, `most_likely_answer`, `responses` — скачивается автоматически.

## 0. Проверка GPU

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print(f"Python: {sys.version}")

## 1. Клон репозитория и зависимости

Клон — секунды; pip — 2-5 минут (transformers + sae-lens + accelerate).

In [ ]:
import os, sys, subprocess

GIT_URL = "https://github.com/SadreevAmir/sae-muc.git"
GIT_BRANCH = "main"
REPO_DIR = "/content/sae-muc"

sae_pkg = os.path.join(REPO_DIR, "sae_muc")
if not os.path.isdir(sae_pkg):
    if os.path.isdir(REPO_DIR):
        subprocess.run(["rm", "-rf", REPO_DIR], check=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_URL, REPO_DIR],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("-> Installing dependencies (may take a few minutes) ...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "numpy>=2.0.0,<2.1",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U", "--no-cache-dir",
    "transformers>=4.40", "accelerate",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "sae-lens>=6.0", "pandas", "tqdm", "jsonlines", "huggingface_hub",
])

import torch
assert torch.cuda.is_available(), "GPU not available — change Runtime type to GPU!"
print(f"\nREPO: {REPO_DIR}")
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. (Опционально) Google Drive

Если файлы лежат на Drive — смонтируйте и укажите пути в секции 4.

In [ ]:
MOUNT_DRIVE = False  # @param {type:"boolean"}

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted at /content/drive")
else:
    print("Drive not mounted — will use file upload in section 3.")

## 3. Загрузка файлов

Загрузите два файла:
- **test.csv** — CSV с колонками `question`, `verbal_uncertainty`, `sentence_semantic_entropy`
- **stats JSON** — `sae_feature_analysis_merged_stats.json`

Если файлы уже есть (Drive / предыдущий запуск) — поставьте `UPLOAD_FILES = False` и укажите пути вручную в секции 4.

In [ ]:
import os

UPLOAD_FILES = True  # @param {type:"boolean"}

UPLOAD_DIR = "/content/uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

_uploaded_csv = None
_uploaded_json = None

if UPLOAD_FILES:
    from google.colab import files

    print("=" * 60)
    print("Step 1/2: Upload test.csv")
    print("  Required columns: question, verbal_uncertainty, sentence_semantic_entropy")
    print("=" * 60)
    up1 = files.upload()
    for fn, data in up1.items():
        dst = os.path.join(UPLOAD_DIR, fn)
        with open(dst, "wb") as f:
            f.write(data)
        if fn.endswith(".csv"):
            _uploaded_csv = dst
            print(f"  -> Saved CSV: {dst}")

    print()
    print("=" * 60)
    print("Step 2/2: Upload stats JSON (sae_feature_analysis_merged_stats.json)")
    print("=" * 60)
    up2 = files.upload()
    for fn, data in up2.items():
        dst = os.path.join(UPLOAD_DIR, fn)
        with open(dst, "wb") as f:
            f.write(data)
        if fn.endswith(".json"):
            _uploaded_json = dst
            print(f"  -> Saved JSON: {dst}")

    if _uploaded_csv:
        print(f"\nCSV:  {_uploaded_csv}")
    if _uploaded_json:
        print(f"JSON: {_uploaded_json}")
else:
    print("Upload skipped — set paths manually in section 4.")

## 4. Конфигурация

Все параметры эксперимента. Если вы загрузили файлы выше, пути заполнятся автоматически.

| Параметр | Описание |
|---|---|
| `TEST_CSV` | CSV с вопросами (question, verbal_uncertainty, sentence_semantic_entropy) |
| `STATS_JSON` | JSON от robust feature analysis |
| `N_ROWS` | Сколько строк из CSV (0 = все) |
| `ALPHA_MAX` | Максимальная сила вмешательства |
| `ALPHA_BINS` | На сколько корзин квантуем alpha (step = max/bins) |
| `TOP_FEATURES` | Сколько top uncertainty-up фич брать из stats JSON |
| `FEATURE_VALUE` | Значение, которое ставится в delta для каждой выбранной фичи |
| `GEN_BATCH_SIZE` | Batch size для генерации (подберите под VRAM) |

In [ ]:
# ── File paths (auto-filled from upload, or set manually) ──
TEST_CSV    = _uploaded_csv  or "/content/uploads/test.csv"       # @param {type:"string"}
STATS_JSON  = _uploaded_json or "/content/uploads/sae_feature_analysis_merged_stats.json"  # @param {type:"string"}

# ── Model ──
MODEL_NAME      = "Mistral-7B-Instruct-v0.3"  # @param {type:"string"}
SAE_RELEASE     = "mistral-7b-res-wg"          # @param {type:"string"}

# ── Experiment ──
N_ROWS          = 0     # @param {type:"integer"} (0 = all rows)
ALPHA_MAX       = 50.0  # @param {type:"number"}
ALPHA_BINS      = 4     # @param {type:"integer"}
TOP_FEATURES    = 32    # @param {type:"integer"}
FEATURE_VALUE   = 1.0   # @param {type:"number"}
GEN_BATCH_SIZE  = 4     # @param {type:"integer"}
SAE_DTYPE       = "float32"  # @param ["float32", "float16", "bfloat16"]

# ── Output ──
OUTPUT_JSONL = "/content/sae_steering_output.jsonl"  # @param {type:"string"}

# ── Validate ──
import os
assert os.path.isfile(TEST_CSV),   f"CSV not found: {TEST_CSV}"
assert os.path.isfile(STATS_JSON), f"JSON not found: {STATS_JSON}"

import pandas as pd
df_preview = pd.read_csv(TEST_CSV)
required_cols = {"question", "verbal_uncertainty", "sentence_semantic_entropy"}
missing = required_cols - set(df_preview.columns)
assert not missing, f"CSV is missing columns: {missing}"

n_total = len(df_preview)
n_use = n_total if N_ROWS == 0 else min(N_ROWS, n_total)
print(f"CSV:        {TEST_CSV}  ({n_total} rows, using {n_use})")
print(f"Stats JSON: {STATS_JSON}")
print(f"Model:      {MODEL_NAME}  (release: {SAE_RELEASE})")
print(f"Alpha:      max={ALPHA_MAX}, bins={ALPHA_BINS}, step={ALPHA_MAX/ALPHA_BINS:.2f}")
print(f"Features:   top {TOP_FEATURES} per layer, value={FEATURE_VALUE}")
print(f"Output:     {OUTPUT_JSONL}")
print()
df_preview.head(5)

## 5. Hugging Face Login

Нужен токен с доступом к модели (для Mistral — accept условия на HF).

In [ ]:
from huggingface_hub import login
login()

## 6. Загрузка модели и SAE

Загружает LLM (~14GB fp16), SAE для каждого слоя из stats, строит delta-вектора.

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sae_lens import SAE

import sys, os
REPO_DIR = "/content/sae-muc"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from sae_muc.hooks import register_sae_latent_hooks, clear_sae_latent_hooks
from sae_muc.layer_map import hf_layers_for_release
from sae_muc.generation import generate_lines_for_batch
from sae_muc.prompts_mini import make_sentence_user_content

torch.manual_seed(42)
np.random.seed(42)

MAX_SE = 2.302585092994045

# ── Resolve model name ──
if "Mistral" in MODEL_NAME:
    full_model_name = f"mistralai/{MODEL_NAME}"
elif "Llama" in MODEL_NAME:
    full_model_name = f"meta-llama/{MODEL_NAME}"
elif "Qwen" in MODEL_NAME:
    full_model_name = f"Qwen/{MODEL_NAME}"
else:
    full_model_name = MODEL_NAME

# ── Load data ──
print("Loading CSV ...")
df = pd.read_csv(TEST_CSV)
n = len(df) if N_ROWS == 0 else min(N_ROWS, len(df))
df = df.iloc[:n].copy()
questions = df["question"].astype(str).tolist()
vu = df["verbal_uncertainty"].to_numpy(dtype=np.float64)
se = df["sentence_semantic_entropy"].to_numpy(dtype=np.float64)

# ── Compute alphas ──
raw_alpha = (se / MAX_SE - vu) * ALPHA_MAX
raw_alpha = np.clip(raw_alpha, 0.0, ALPHA_MAX)
alpha_step = ALPHA_MAX / float(ALPHA_BINS)
alphas = np.clip(np.round(raw_alpha / alpha_step) * alpha_step, 0.0, ALPHA_MAX)
alphas = np.round(alphas, 6)

print(f"Questions: {len(questions)}")
print(f"Alpha step: {alpha_step:.4f}")
print(f"Unique alpha groups: {len(np.unique(alphas))}")
print(f"Alpha distribution: {dict(zip(*np.unique(alphas, return_counts=True)))}")

# ── Load robust features ──
print("\nLoading robust feature stats ...")
with open(STATS_JSON, "r") as f:
    stats = json.load(f)

expected_layers = {l for l, _ in hf_layers_for_release(SAE_RELEASE)}
layer_to_sae_id = {}
layer_to_feature_idx = {}

for item in stats.get("layers", []):
    if "layer" not in item or "sae_id" not in item:
        continue
    layer = int(item["layer"])
    if layer not in expected_layers:
        continue
    feat_idx = item.get("top_uncertainty_feature_idx", [])
    if not feat_idx:
        continue
    layer_to_sae_id[layer] = str(item["sae_id"])
    layer_to_feature_idx[layer] = feat_idx[:TOP_FEATURES]
    print(f"  Layer {layer}: {len(feat_idx[:TOP_FEATURES])} features, sae_id={item['sae_id']}")

assert layer_to_feature_idx, "No layers/features found in stats JSON!"

# ── Load SAEs and build delta vectors ──
print("\nLoading SAEs ...")
layer_to_sae = {}
layer_to_delta = {}

for layer, sae_id in layer_to_sae_id.items():
    t0 = time.time()
    sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=sae_id, device="cpu", dtype=SAE_DTYPE)
    d = torch.zeros(sae.cfg.d_sae, dtype=torch.float32)
    idx = torch.tensor(layer_to_feature_idx[layer], dtype=torch.long)
    d[idx] = float(FEATURE_VALUE)
    d = d / (d.norm() + 1e-8)
    layer_to_sae[layer] = sae
    layer_to_delta[layer] = d
    nz = int((d != 0).sum().item())
    print(f"  Layer {layer}: d_sae={sae.cfg.d_sae}, {nz} active features, loaded in {time.time()-t0:.1f}s")

hook_layers = sorted(layer_to_sae.keys())
messages = [[{"role": "user", "content": make_sentence_user_content(q)}] for q in questions]

# ── Load LLM ──
print(f"\nLoading {full_model_name} (fp16, device_map=auto) ...")
from transformers import AutoModelForCausalLM, AutoTokenizer

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    full_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(full_model_name, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Model loaded in {time.time()-t0:.1f}s")
print(f"\nReady: {len(questions)} questions, {len(hook_layers)} SAE layers, alpha in [{0}, {ALPHA_MAX}]")

## 7. Генерация

Группирует вопросы по значению alpha, для каждой группы регистрирует хуки и генерирует ответы.

- `alpha=0` — baseline (без хуков, но с реальной генерацией)
- `alpha>0` — SAE steering с соответствующей силой

In [ ]:
from tqdm.auto import tqdm

by_alpha = {}
for i, a in enumerate(alphas.tolist()):
    by_alpha.setdefault(float(a), []).append(i)

results = [None] * len(questions)
batch_size = max(1, GEN_BATCH_SIZE)

print(f"Generating for {len(by_alpha)} alpha groups ...\n")

for alpha in sorted(by_alpha.keys()):
    clear_sae_latent_hooks(model)
    if alpha > 0:
        register_sae_latent_hooks(model, layer_to_sae, layer_to_delta, hook_layers, alpha)

    idxs = by_alpha[alpha]
    desc = f"alpha={alpha:.4f} ({len(idxs)} qs)"

    for j in tqdm(range(0, len(idxs), batch_size), desc=desc):
        chunk_idx = idxs[j : j + batch_size]
        batch_q = [questions[k] for k in chunk_idx]
        batch_m = [messages[k] for k in chunk_idx]
        lines = generate_lines_for_batch(model, tokenizer, batch_q, batch_m, alpha)
        for local_i, global_i in enumerate(chunk_idx):
            results[global_i] = lines[local_i]

clear_sae_latent_hooks(model)

assert all(r is not None for r in results), "Some rows were not generated!"
print(f"\nDone: {len(results)} rows generated.")

## 8. Сохранение и скачивание результатов

In [ ]:
import json
from pathlib import Path

out_path = Path(OUTPUT_JSONL)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w", encoding="utf-8") as f:
    for item in results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved: {out_path}  ({len(results)} lines)")

# Preview
print("\n" + "=" * 60)
print("Preview (first 3 rows):")
print("=" * 60)
for i, item in enumerate(results[:3]):
    print(f"\n[{i}] alpha={item['alpha']:.4f}")
    print(f"    Q: {item['question'][:80]}")
    ans = item['most_likely_answer'][:120] if item['most_likely_answer'] else '(empty)'
    print(f"    A: {ans}")
    print(f"    #responses: {len(item['responses'])}")

# Stats
print("\n" + "=" * 60)
alpha_vals = [r["alpha"] for r in results]
print(f"Total rows:     {len(results)}")
print(f"Baseline (a=0): {sum(1 for a in alpha_vals if a == 0)}")
print(f"Steered (a>0):  {sum(1 for a in alpha_vals if a > 0)}")

# Download
try:
    from google.colab import files
    files.download(str(out_path))
    print(f"\nDownload started: {out_path.name}")
except ImportError:
    print(f"\nNot in Colab — file saved at: {out_path}")

## 9. (Опционально) Быстрый анализ результатов

Простая статистика по сгенерированным ответам: длина ответов, распределение alpha.

In [ ]:
import pandas as pd
import numpy as np

df_out = pd.DataFrame(results)
df_out["answer_len"] = df_out["most_likely_answer"].str.len()
df_out["n_responses"] = df_out["responses"].apply(len)

print("Per-alpha group stats:")
print(df_out.groupby("alpha").agg(
    count=("question", "count"),
    mean_answer_len=("answer_len", "mean"),
    mean_n_responses=("n_responses", "mean"),
).to_string())

print(f"\nOverall: {len(df_out)} rows")
print(f"Answer length: mean={df_out['answer_len'].mean():.0f}, "
      f"min={df_out['answer_len'].min()}, max={df_out['answer_len'].max()}")